# P1 — Gobierno de datos para Big Data

**Big Data — 7BM1 — 26-27/1**  
**Sesión 7 — 08/09/2026 — 90 min**

### Equipo
- Integrante 1: Adair Hernandez Valdivia
- Integrante 2: Kitzia Maria Araujo Perez
- Integrante 3: Luis Axel Zarate Lozano

## Pregunta central

> **¿Estos datos son suficientemente confiables y gobernables para utilizarlos en la aplicación propuesta de estimación de demanda y apoyo a decisiones operativas?**

El código es un instrumento de observación. La práctica evalúa principalmente la calidad de la evidencia, su interpretación, la relación problema → riesgo → regla de gobierno y la decisión final.


## 0. Archivos y reglas del caso

Archivos esperados en la misma carpeta del notebook:

- `P1_viajes.csv`
- `P1_estaciones.csv`

Reglas conocidas:

- unidades válidas: `U01`–`U12`;
- estaciones válidas: las presentes en `P1_estaciones.csv`;
- capacidad máxima: **80 pasajeros**;
- servicios previstos: `REGULAR` y `EXPRES`;
- un registro debería recibirse en **≤ 10 min** desde su inicio;
- las coordenadas de origen deben ser coherentes con la estación registrada;
- `viaje_id` debe identificar inequívocamente un viaje.

**Importante:** `ocupacion` aparece en el extracto, pero su unidad/escala no está documentada.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

viajes = pd.read_csv('P1_viajes.csv')
estaciones = pd.read_csv('P1_estaciones.csv')

print('viajes:', viajes.shape)
print('estaciones:', estaciones.shape)
display(viajes.head())
display(estaciones)

viajes: (60, 13)
estaciones: (8, 6)


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id
0,V0001,2026-09-03 06:00:00,2026-09-03 06:02:00,U01,E01,E04,19.4800,-99.1200,REGULAR,19,31,VALIDADOR,C001
1,V0002,2026-09-03 06:12:00,2026-09-03 06:15:00,U02,E02,E05,19.4325,-99.1332,REGULAR,26,42,INTEGRACION,C002
2,V0003,2026-09-03 06:24:00,2026-09-03 06:28:00,U03,E03,E06,19.4100,-99.0700,REGULAR,33,53,GPS_APP,C003
3,V0004,2026-09-03 06:36:00,2026-09-03 06:41:00,U04,E04,E07,19.3500,-99.1500,EXPRES,40,64,VALIDADOR,C004
4,V0005,2026-09-03 06:48:00,2026-09-03 06:54:00,U05,E05,E08,19.4200,-99.2100,REGULAR,47,75,NaN,C005


,estacion_id,nombre,lat_ref,lon_ref,zona,activa
0,E01,Estación Norte,19.4800,-99.1200,Norte,True
1,E02,Estación Centro,19.4325,-99.1332,Centro,True
2,E03,Estación Oriente,19.4100,-99.0700,Oriente,True
3,E04,Estación Sur,19.3500,-99.1500,Sur,True
4,E05,Estación Poniente,19.4200,-99.2100,Poniente,True
5,E06,Estación Universidad,19.3320,-99.1870,Sur,True
6,E07,Estación Mercado,19.4450,-99.1050,Centro,True
7,E08,Estación Terminal,19.5000,-99.1600,Norte,True


## Fase 1 — Comprender el ecosistema de datos (≈10 min)

Complete brevemente:

| Elemento | Identificación del equipo |
|---|---|
| Fuente(s) | Registro interno de la caja negra del camion asi como registros de bitacora |
| Dataset(s) | P1_estaciones.csv y P1_viajes.csv |
| Forma de almacenamiento recibida | Dos CSV con tablas estructuradas |
| Uso propuesto | Predicción de retrasos |
| Quién necesita confiar en los datos | Encargados de lineas y usuarios del transporte |
| Metadatos mínimos que deberían conservarse | Sensores, GPS, timestamps |

**Pregunta:** ¿qué relación existe entre la calidad del dataset y el valor que la organización espera obtener?

**Respuesta del equipo:**  
Directamente proporcional esto se debe a que entre mejor sean los datos la calidad de las predicciones sera mayor y la estrategias que se implementen seran mas efectivas


## Fase 2 — Perfilado del dataset (≈20 min)

Obtenga evidencia sobre:

1. dimensiones y tipos;
2. faltantes;
3. duplicados;
4. valores únicos y frecuencias;
5. mínimos/máximos;
6. fechas y oportunidad;
7. compatibilidad con catálogos/reglas conocidas.

No limpie todo el dataset. Primero **observe y documente**.


In [2]:
# 2.2 Valores faltantes
df = pd.DataFrame(viajes.isnull().sum())
display(df)


,0
viaje_id,0
fecha_hora_inicio,3
fecha_hora_recepcion,0
unidad_id,0
estacion_origen_id,0
estacion_destino_id,0
lat_origen,0
lon_origen,0
tipo_servicio,0
pasajeros,0


In [3]:
# Si alguna cadena vacía no fue interpretada como NaN, puede comprobarla así:
df = pd.DataFrame((viajes == '').sum())
display(df)

,0
viaje_id,0
fecha_hora_inicio,0
fecha_hora_recepcion,0
unidad_id,0
estacion_origen_id,0
estacion_destino_id,0
lat_origen,0
lon_origen,0
tipo_servicio,0
pasajeros,0


In [7]:
# 2.3 Duplicados exactos
print('Filas duplicadas exactamente:', viajes.duplicated().sum())


frecuencias_ids = viajes['viaje_id'].value_counts()
df = frecuencias_ids[frecuencias_ids > 1].rename('frecuencia').to_frame()
display(df.head(10))

Filas duplicadas exactamente: 1


,frecuencia
viaje_id,
V0025,2
V0011,2


In [8]:
# 2.4 Frecuencias de variables categóricas

df = pd.DataFrame(viajes['tipo_servicio'].value_counts())
display(df)
df = pd.DataFrame(viajes['fuente_registro'].value_counts())
display(df)
df = pd.DataFrame(viajes['estacion_origen_id'].value_counts().head(10))
display(df)
df = pd.DataFrame(viajes['estacion_destino_id'].value_counts().head(10))
display(df)
df = pd.DataFrame(viajes['unidad_id'].value_counts().head(10))
display(df)


,count
tipo_servicio,
REGULAR,42
EXPRES,14
Regular,1
regular,1
Expres,1
EXPRESS,1


,count
fuente_registro,
VALIDADOR,19
INTEGRACION,19
GPS_APP,19


,count
estacion_origen_id,
E03,9
E01,8
E02,8
E04,7
E05,7
E07,7
E08,7
E06,6
E99,1


,count
estacion_destino_id,
E04,8
E05,8
E06,8
E07,7
E08,7
E01,7
E02,7
E03,7
E00,1


,count
unidad_id,
U11,6
U01,5
U03,5
U04,5
U05,5
U02,5
U06,5
U08,5
U10,5


In [ ]:
# 2.5 Rangos numéricosribe()
resumen = {
    'pasajeros': [],
    'ocupacion': [],
    'lat_origen': [],
    'lon_origen': []
}

# resumen estadístico de las variables numéricas
for col in ['pasajeros', 'ocupacion', 'lat_origen', 'lon_origen']:
    resumen[col] = {
        'count': viajes[col].count(),
        'mean': viajes[col].mean(),
        'std': viajes[col].std(),
        'min': viajes[col].min(),
        '25%': viajes[col].quantile(0.25),
        '50%': viajes[col].quantile(0.5),
        '75%': viajes[col].quantile(0.75),
        'max': viajes[col].max(),
    }
df_resumen = pd.DataFrame(resumen)
display(df_resumen)

,pasajeros,ocupacion,lat_origen,lon_origen
count,60.000000,60.000000,60.000000,60.000000
mean,39.450000,56.383333,22.289942,-99.137797
std,19.045374,21.462693,22.202107,0.043176
min,-3.000000,21.000000,19.332000,-99.210000
25%,24.750000,37.750000,19.410000,-99.160000
50%,38.000000,56.500000,19.432500,-99.133200
75%,53.250000,72.750000,19.453750,-99.105000
max,96.000000,93.000000,191.400000,-99.070000


### Fechas

Los registros pueden contener representaciones inconsistentes. Procure transformar las fechas **sin detener el notebook ante un valor no interpretable**. Una opción es usar `errors='coerce'` y revisar qué registros se convierten en `NaT`.

> No asuma que un valor que no pudo convertirse está necesariamente “mal” sin revisar el dato original y el contexto.


In [23]:
# 2.6 Fechas: complete o adapte este bloque.
# Sugerencia: pruebe primero la conversión y compare el resultado con la columna original.

try:
    inicio_parseado = pd.to_datetime(
        viajes['fecha_hora_inicio'], format='mixed', errors='coerce', dayfirst=True
    )
    # print(inicio_parseado)
except TypeError:
    # Compatibilidad con versiones anteriores de pandas.
    inicio_parseado = viajes['fecha_hora_inicio'].apply(
        lambda x: pd.to_datetime(x, errors='coerce', dayfirst=True)
    )
    # print(inicio_parseado)

recepcion_parseada = pd.to_datetime(viajes['fecha_hora_recepcion'], errors='coerce')

print('Inicio no interpretable / ausente:', inicio_parseado.isna().sum())

# TODO: calcule el retraso de recepción en minutos y localice registros > 10 min.
df_diferencia = pd.DataFrame({'inicio': inicio_parseado, 'recepcion': recepcion_parseada})
retraso = (df_diferencia['recepcion'] - df_diferencia['inicio']).dt.total_seconds() / 60
print('Retraso de recepción (minutos):')
df = pd.DataFrame(retraso.head(10), columns=['retraso_minutos'])
display(df)

# Localizar registros con retraso > 10 minutos
registros_retraso = viajes[retraso > 10]
df_retraso = pd.DataFrame(registros_retraso.head(10))
display(df_retraso)



Inicio no interpretable / ausente: 4
Retraso de recepción (minutos):


,retraso_minutos
0,2.0
1,3.0
2,4.0
3,5.0
4,6.0
5,7.0
6,NaN
7,2.0
8,3.0
9,4.0


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id
15,V0016,2026-09-03 09:00:00,2026-09-03 09:17:00,U04,E08,E03,19.50,-99.16,EXPRES,66,44,VALIDADOR,C016
40,V0041,2026-09-03 14:00:00,2026-09-03 14:25:00,U05,E01,E04,19.48,-99.12,REGULAR,67,91,INTEGRACION,C005
58,V0059,2026-09-03 17:36:00,2026-09-03 18:39:00,U11,E03,E06,19.41,-99.07,REGULAR,19,61,INTEGRACION,C005


In [31]:
# 2.7 Reglas y catálogos
unidades_validas = {f'U{i:02d}' for i in range(1, 13)}
estaciones_validas = set(estaciones['estacion_id'])
servicios_validos = {'REGULAR', 'EXPRES'}

# TODO: utilice estas referencias para localizar valores que no cumplen las reglas.
# Ejemplo de patrón (adáptelo a la columna que necesite):
# viajes[~viajes['...'].isin(...)]
df_invalid_unidades = pd.DataFrame(viajes[~viajes['unidad_id'].isin(unidades_validas)])
print('Unidades no válidas:')
display(df_invalid_unidades)

df_estaciones_invalidas = pd.DataFrame(viajes[~viajes['estacion_origen_id'].isin(estaciones_validas)])
print('Estaciones de origen no válidas:')
display(df_estaciones_invalidas)

df_servicios_invalidos = pd.DataFrame(viajes[~viajes['tipo_servicio'].isin(servicios_validos)])
print('Tipos de servicio no válidos:')
display(df_servicios_invalidos)


Unidades no válidas:


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id
18,V0019,2026-09-03 09:36:00,2026-09-03 09:42:00,U99,E03,E06,19.41,-99.07,REGULAR,29,77,VALIDADOR,C001
44,V0045,2026-09-03 14:48:00,2026-09-03 14:52:00,UX3,E05,E08,19.42,-99.21,REGULAR,37,59,GPS_APP,C009


Estaciones de origen no válidas:


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id
21,V0022,2026-09-03 10:12:00,2026-09-03 10:14:00,U10,E99,E01,19.332,-99.187,REGULAR,50,34,VALIDADOR,C004


Tipos de servicio no válidos:


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id
8,V0009,2026-09-03 07:36:00,2026-09-03 07:39:00,U09,E01,E04,19.4800,-99.1200,Regular,17,43,GPS_APP,C009
26,V0027,2026-09-03 11:12:00,2026-09-03 11:19:00,U03,E03,E06,19.4100,-99.0700,regular,27,89,GPS_APP,C009
33,V0034,2026-09-03 12:36:00,2026-09-03 12:43:00,U10,E02,E05,19.4325,-99.1332,Expres,18,90,VALIDADOR,C016
57,V0058,2026-09-03 17:24:00,2026-09-03 17:27:00,U10,E02,E05,19.4325,-99.1332,EXPRESS,12,50,VALIDADOR,C004


In [33]:
# 2.8 Comparación con el catálogo maestro de estaciones
# Puede combinar viajes con estaciones usando estacion_origen_id ↔ estacion_id.
# Después compare lat_origen/lon_origen con lat_ref/lon_ref.

# TODO: construya la comparación y conserve evidencia de los casos que le parezcan relevantes.
viajes_con_estaciones = viajes.merge(estaciones, left_on='estacion_origen_id', right_on='estacion_id', how='left')

comparacion = viajes_con_estaciones[(viajes_con_estaciones['lat_origen'] != viajes_con_estaciones['lat_ref']) | (viajes_con_estaciones['lon_origen'] != viajes_con_estaciones['lon_ref'])]
print('Comparación de coordenadas (lat/lon) con el catálogo maestro de estaciones:')
display(comparacion)


Comparación de coordenadas (lat/lon) con el catálogo maestro de estaciones:


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id,estacion_id,nombre,lat_ref,lon_ref,zona,activa
21,V0022,2026-09-03 10:12:00,2026-09-03 10:14:00,U10,E99,E01,19.3320,-99.1870,REGULAR,50,34,VALIDADOR,C004,NaN,NaN,NaN,NaN,NaN,NaN
23,V0024,2026-09-03 10:36:00,2026-09-03 10:40:00,U12,E08,E03,19.4450,-99.1050,EXPRES,64,56,GPS_APP,C006,E08,Estación Terminal,19.50,-99.16,Norte,True
35,V0036,2026-09-03 13:00:00,2026-09-03 13:02:00,U12,E04,E07,191.4000,-99.1500,EXPRES,32,36,GPS_APP,C018,E04,Estación Sur,19.35,-99.15,Sur,True
51,V0052,2026-09-03 16:12:00,2026-09-03 16:16:00,U04,E04,E07,19.4325,-99.1332,EXPRES,28,60,VALIDADOR,C016,E04,Estación Sur,19.35,-99.15,Sur,True


## Fase 3 — Auditoría de calidad (≈20 min)

Registre **al menos seis hallazgos no redundantes**. No gana puntos por acumular filas sin interpretación.

Dimensiones trabajadas: **completitud, validez, consistencia, oportunidad y exactitud**.

- Use **exactitud** sólo si existe una referencia que permita contrastar el dato.
- Si la evidencia no permite determinar algo, escriba **“no puede determinarse con la evidencia disponible”**.

Complete la lista y ejecute la celda para visualizar su matriz.


In [ ]:
hallazgos = [
    # Ejemplo de estructura (NO corresponde a una respuesta del dataset):
    # {
    #     'Hallazgo': '...',
    #     'Dimension_criterio': '...',
    #     'Evidencia': '...',
    #     'Impacto': '...'
    # },
]

matriz_hallazgos = pd.DataFrame(hallazgos)
matriz_hallazgos

""


## Fase 4 — Gobierno, seguridad y ética (≈20 min)

Para esta práctica, **gobierno de datos** significa establecer responsabilidades, metadatos y reglas para decidir cómo se reciben, validan, utilizan y protegen los datos.

Proponga **al menos cuatro reglas/controles**, incluyendo:

1. una de validación;
2. una de metadatos/procedencia;
3. una de acceso o protección;
4. una derivada directamente de otro hallazgo.

Analice también:

- quién debería ser responsable de cada regla;
- en qué momento debería aplicarse;
- qué atributos necesitan realmente los usuarios de la aplicación;
- si algún atributo debería eliminarse, restringirse o protegerse por no ser necesario para el propósito.


In [ ]:
reglas_gobierno = [
    # {
    #     'Problema_riesgo': '...',
    #     'Regla_control': '...',
    #     'Responsable': '...',
    #     'Momento_aplicacion': '...'
    # },
]

matriz_gobierno = pd.DataFrame(reglas_gobierno)
matriz_gobierno

### Preguntas de análisis de acceso y uso responsable

1. ¿`conductor_id` es necesario para estimar demanda y apoyar la decisión operativa planteada? ¿Qué harían con esa variable y por qué?

**Respuesta:**  

2. ¿La ausencia de `fuente_registro` afecta sólo la calidad o también la gobernanza del dato? Explique.

**Respuesta:**  

3. ¿Qué información adicional necesitarían antes de afirmar que `ocupacion` está bien o mal registrada?

**Respuesta:**  



## Fase 5 — Decisión final (≈15 min)

Seleccione una opción:

- [ ] **Sí**
- [ ] **Sí, con condiciones**
- [ ] **No todavía**

### Conclusión (aprox. 150–200 palabras)

Incluya:

- al menos **tres evidencias** obtenidas durante la auditoría;
- problemas principales;
- riesgos para la aplicación propuesta;
- controles imprescindibles;
- alcance de la autorización.

**Conclusión del equipo:**  



## Reflexión final — sin puntaje independiente

> **¿Que un dataset sea públicamente accesible significa automáticamente que sea confiable, de calidad y apropiado para cualquier uso?**

**Respuesta breve:**  



## Checklist antes de entregar

- [ ] El equipo está identificado.
- [ ] Las celdas relevantes conservan sus salidas.
- [ ] Hay al menos 6 hallazgos no redundantes con evidencia e impacto.
- [ ] Hay al menos 4 reglas/controles de gobierno.
- [ ] La decisión final usa evidencia del propio notebook.
- [ ] No se atribuyó “exactitud” sin una referencia suficiente.
- [ ] El archivo se guardó como `P1_EquipoXX_GobiernoDatos.ipynb`.
- [ ] Un integrante entregará el notebook en Google Classroom según la hora límite indicada por el docente.
